# What iBART can (and can't) find

iBART discovers **nonlinear equations** -- that is its whole purpose. It builds
nonlinear descriptors by composing operators (`square, sqrt, log, exp, sin, cos,
abs, *, /, |a-b|`) on the inputs. The model is *linear in the coefficients on a
nonlinear basis* (like SINDy or polynomial regression): the discovered equation
is nonlinear, the fit is convex.

This notebook shows the wins, the one real limitation, and two workarounds.

## 1. Nonlinear expressions it recovers exactly

In [1]:
import numpy as np
from jax_ibart import IBart

rng = np.random.default_rng(0)
n = 200
x1, x2, x3 = (rng.uniform(0.1, 1, n) for _ in range(3))
X = np.column_stack([x1, x2, x3]); names = ["x1", "x2", "x3"]

def r2(model, y):
    p = model.predict(X)
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

# transcendental: y = sin(pi * x1 * x2)
y = np.sin(np.pi * x1 * x2)
m = IBart(backend="diff", schedule=("b", "u"), max_dim=1, seed=0, steps=1500).fit(X, y, names)
print(f"sin(pi*x1*x2)         -> {m.descriptors_}  R2={r2(m, y):.4f}")

# the OIS paper's example: y = (exp(x1) - exp(x2))^2
y = (np.exp(x1) - np.exp(x2)) ** 2
m = IBart(backend="diff", schedule=("u", "b", "u"), max_dim=1, seed=0, steps=1500).fit(X, y, names)
print(f"(exp(x1)-exp(x2))^2   -> {m.descriptors_}  R2={r2(m, y):.4f}")

sin(pi*x1*x2)         -> [sin(pi*x1*x2)]  R2=1.0000


(exp(x1)-exp(x2))^2   -> [Abs(exp(x1) - exp(x2))**2]  R2=1.0000


Both are recovered with R^2 = 1.0. Transcendental, rational, product, and
composed nonlinearities are all in scope as long as the operators can build them.

## 2. The one real limitation: free parameters *inside* a nonlinearity

iBART fits **linear coefficients** on a fixed, operator-generated library. It
cannot fit a *continuous parameter buried inside* a term unless that exact term
is generated. A power law with a non-integer exponent is the classic case: there
is no arbitrary-power operator, so `x^2.3` is only *approximated* (here by `x^2`).

In [2]:
xp = rng.uniform(0.5, 3.0, n)
Xp = np.column_stack([xp, rng.uniform(0, 1, n)])
y = xp ** 2.3                      # exponent 2.3 is not constructible
m = IBart(backend="diff", schedule=("u",), max_dim=1, seed=0, steps=1200).fit(Xp, y, ["x", "d"])
pr = m.predict(Xp)
print("x^2.3  -> found", m.descriptors_,
      f" R2={1 - ((y-pr)**2).sum()/((y-y.mean())**2).sum():.4f}  (approximate, not exact)")

x^2.3  -> found [x**2]  R2=0.9979  (approximate, not exact)


Same story for **Arrhenius** `k = A exp(-E/RT)` with a fitted activation energy
`E` *inside* the exponential, or **Michaelis-Menten** `v = Vmax S/(Km+S)` with a
fitted `Km` *in the denominator*. These are nonlinear *in the parameters*. Two
standard fixes:

### Workaround A -- linearize (Arrhenius)

In [3]:
R = 8.314; A_true = 1e3; E_true = 50000.0
T = rng.uniform(300, 600, n)
k = A_true * np.exp(-E_true / (R * T))

# ln(k) = ln(A) - (E/R) * (1/T)  is LINEAR in 1/T
invT = 1.0 / T
slope, intercept = np.polyfit(invT, np.log(k), 1)
print(f"recovered E = {-slope * R:,.0f} J/mol   (true 50000)")
print(f"recovered A = {np.exp(intercept):.1f}      (true 1000)")

recovered E = 50,000 J/mol   (true 50000)
recovered A = 1000.0      (true 1000)


Transforming to `ln k` vs `1/T` turns the nonlinear-in-parameter problem into a
linear one. iBART recovers the same structure when given `1/T` (its `inv`
operator) and the log-transformed target.

### Workaround B -- implicit / null-space form (Michaelis-Menten)

**What "null space" means, basically.** Usual regression fits `y = f(x)` -- one
column is the target, and we predict it from the others. But a constraint like
`x^2 + y^2 = 1` has *no* target: no single variable is "the output." Instead, a
particular weighted sum of the candidate terms is **always zero**.

So build a table whose columns are candidate terms (here `S, v, S^2, v^2, S*v`,
and a column of `1`s). We want weights `w` such that

    w1*S + w2*v + ... + w6*1  =  0   for every row.

That weight vector is called a **null-space vector** of the table: multiplying
the table by it gives all zeros. Standard regression can't find it (there's no
`y`); a singular value decomposition (SVD) can -- the direction the data *never*
varies along is exactly the constraint. The smallest singular value measures how
close to "exactly zero" it is (0 = an exact law).

For Michaelis-Menten, rearranging `v = Vmax*S/(Km+S)` gives
`Vmax*S - Km*v - S*v = 0` -- a sum of terms that is always zero, i.e. a
null-space vector of the `[S, v, S*v, ...]` table. That is what we recover:

In [4]:
from jax_ibart.dynamics import implicit_constraints
import sympy as sp

Vmax, Km = 5.0, 2.0
S = rng.uniform(0.1, 10.0, n)
v = Vmax * S / (Km + S)

# v(Km + S) = Vmax S  ->  Vmax*S - Km*v - S*v = 0  : an algebraic constraint
res = implicit_constraints(np.column_stack([S, v]), ["S", "v"], degree=2, tol=1e-2)
print("recovered constraint:", sp.nsimplify(res["constraint"], rational=False).evalf(3))
print("residual (0 = exact):", round(res["residual"], 6))
print("\ntrue: 5*S - 2*v - S*v = 0  ->  divide by 5:  S - 0.4*v - 0.2*S*v = 0")

recovered constraint: -0.2*S*v + S - 0.4*v
residual (0 = exact): 0.0

true: 5*S - 2*v - S*v = 0  ->  divide by 5:  S - 0.4*v - 0.2*S*v = 0


The implicit (null-space) formulation recovers the rational Michaelis-Menten
kinetics **exactly** as an algebraic constraint, from which `Vmax` and `Km` read
straight off the coefficients. This is the same machinery that underlies the
DAE foundation (`implicit_constraints`) and SINDy-PI (see
[references](../references.md)).

**Summary:** nonlinear *expressions* -- yes, that is the point. Free parameters
*inside* a nonlinearity -- only via the operator library, a linearizing
transform, or the implicit null-space form.